In [95]:
from langchain_ollama import ChatOllama
from langchain_core.tools import tool, InjectedToolArg
from langchain_core.messages import HumanMessage, ToolMessage
import requests
from typing import Annotated

In [ ]:
llm = ChatOllama(
    model = "llama3.2:3b",
    temperature=0.4
)

In [97]:
@tool
def get_exchange_rate(from_currency: str, to_currency: str) -> float:
    """
    Get exchange rate between two currencies.
    Example: USD -> INR
    """
    try:
        url = f"https://api.exchangerate-api.com/v4/latest/{from_currency.upper()}"
        response = requests.get(url, timeout=5)

        if response.status_code != 200:
            raise Exception(f"API Error: {response.status_code}")

        data = response.json()

        if to_currency.upper() not in data['rates']:
            raise Exception(f"Invalid target currency: {to_currency}")

        return data['rates'][to_currency.upper()]

    except Exception as e:
        print(f"Error: {e}")
        return None
    
@tool
def convert_currency(from_currency_amount: int, exchange_rate: Annotated[float, InjectedToolArg]) -> float:
    """
    Convert currency using the exchange rate.
    """
    try:
        if exchange_rate is None:
            raise Exception("Invalid exchange rate")

        return from_currency_amount * exchange_rate

    except Exception as e:
        print(f"Error: {e}")
        return None

In [98]:
get_exchange_rate.invoke({
    "from_currency": "USD",
    "to_currency": "INR"
})

convert_currency.invoke({
    "from_currency_amount": 100,
    "exchange_rate": 82.5
})

8250.0

In [99]:
print(convert_currency.args)
print(convert_currency.name)
print(convert_currency.description)

{'from_currency_amount': {'title': 'From Currency Amount', 'type': 'integer'}}
convert_currency
Convert currency using the exchange rate.


In [100]:
llm_with_tools = llm.bind_tools([get_exchange_rate, convert_currency])

In [101]:
query = HumanMessage(content="Give me the exchange rate from USD to INR and convert 100 USD to INR.")
messages = [query]

In [102]:
result = llm_with_tools.invoke(messages)
messages.append(result)

ResponseError: registry.ollama.ai/library/tinyllama:latest does not support tools (status code: 400)

In [83]:
result.tool_calls

[{'name': 'get_exchange_rate',
  'args': {'from_currency': 'USD', 'to_currency': 'INR'},
  'id': 'f27845ea-6bdd-477a-9ea7-8df46d9e6f26',
  'type': 'tool_call'},
 {'name': 'convert_currency',
  'args': {'from_currency_amount': '100'},
  'id': '9d9408a0-d0d0-4f36-a08f-46155d2c6827',
  'type': 'tool_call'}]

In [84]:
exchange_rate = None

for tool_call in result.tool_calls:
    name = tool_call["name"]
    args = tool_call["args"]
    tool_call_id = tool_call["id"]

    if name == "get_exchange_rate":
        exchange_rate = get_exchange_rate.invoke(args)

        tool_msg = ToolMessage(
            content=str(exchange_rate),
            tool_call_id=tool_call_id
        )

        messages.append(tool_msg)

    elif name == "convert_currency":
        args["exchange_rate"] = exchange_rate
        args["from_currency_amount"] = float(args["from_currency_amount"])

        result = convert_currency.invoke(args)

        tool_msg = ToolMessage(
            content=str(result),
            tool_call_id=tool_call_id
        )

        messages.append(tool_msg)

In [85]:
messages

[HumanMessage(content='Give me the exchange rate from USD to INR and convert 100 USD to INR.', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-04-22T07:17:08.26770342Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5596647012, 'load_duration': 315201641, 'prompt_eval_count': 219, 'prompt_eval_duration': 133533288, 'eval_count': 46, 'eval_duration': 5012292508, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019db40c-ac4c-7971-bf5b-9348560354a1-0', tool_calls=[{'name': 'get_exchange_rate', 'args': {'from_currency': 'USD', 'to_currency': 'INR'}, 'id': 'f27845ea-6bdd-477a-9ea7-8df46d9e6f26', 'type': 'tool_call'}, {'name': 'convert_currency', 'args': {'from_currency_amount': 100.0, 'exchange_rate': 93.61}, 'id': '9d9408a0-d0d0-4f36-a08f-46155d2c6827', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 219, 'out

In [86]:
result = llm_with_tools.invoke(messages)

print(result.content)

The current exchange rate from USD to INR is 1 USD = 93.61 INR.

Converting 100 USD to INR, we get:

100 USD x 93.61 INR/USD = 9361.0 INR


In [32]:
messages.append(tool_result)

In [33]:
messages

[HumanMessage(content='346 INR is equals to how many GBP?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'llama3.2:3b', 'created_at': '2026-04-20T13:25:16.168520047Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3758924789, 'load_duration': 199560125, 'prompt_eval_count': 172, 'prompt_eval_duration': 470660843, 'eval_count': 32, 'eval_duration': 3011940709, 'logprobs': None, 'model_name': 'llama3.2:3b', 'model_provider': 'ollama'}, id='lc_run--019dab11-0444-7f02-889b-dff448a32f3c-0', tool_calls=[{'name': 'convert_currency', 'args': {'from_currency': 'INR', 'to_currency': 'GBP', 'amount': '346'}, 'id': '56961f16-b5db-4c26-9df4-e62bbfa5e093', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 172, 'output_tokens': 32, 'total_tokens': 204}),
 ToolMessage(content='2.76', name='convert_currency', tool_call_id='56961f16-b5db-4c26-9df4-e62bbfa5e093')]

In [36]:
final_response = llm_with_tools.invoke(messages)

In [37]:
print(final_response.content)

The amount 346 INR is equivalent to approximately 2.76 GBP.
